# EDA: train vs test スペクトル比較 (Asao)

**目的:** test 種 6 種は train 13 種に対してスペクトル空間上どこに位置するか可視化し、未知種への汎化戦略を立てる。

**仮説の方向:**
- test 種が train 種クラスタの内側 → 既存モデルで素直に予測可
- test 種が外側 → ドメイン適応 / 強い正則化 / SNV-MSC 系の前処理が効くはず

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.competition_io import read_csv_with_fallback, find_spectral_columns

# 日本語フォント設定（システム installed の Noto Sans CJK JP を使う）
_JP_FONT_TTC = '/usr/share/fonts/google-noto-cjk/NotoSansCJK-Regular.ttc'
if Path(_JP_FONT_TTC).exists():
    fm.fontManager.addfont(_JP_FONT_TTC)
sns.set_theme(style='whitegrid')
plt.rcParams['font.family'] = ['Noto Sans CJK JP', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

FIG_DIR = ROOT / 'data/figure/eda_asao'
FIG_DIR.mkdir(parents=True, exist_ok=True)

train = read_csv_with_fallback(ROOT / 'data/raw/train.csv')
test = read_csv_with_fallback(ROOT / 'data/raw/test.csv')
spectral_columns = find_spectral_columns(train.columns)
wavelengths = np.array([float(c) for c in spectral_columns])
print('train', train.shape, 'test', test.shape, 'wavelengths', wavelengths.shape)

## 1. 種ごとの平均スペクトル

サンプル個体差を均してから比較するため、種ごとに平均スペクトルを算出する。

In [ ]:
def mean_spectrum_per_species(df: pd.DataFrame) -> pd.DataFrame:
    return df.groupby(['species number', '樹種'])[spectral_columns].mean().reset_index()

train_mean = mean_spectrum_per_species(train)
test_mean = mean_spectrum_per_species(test)
print('train species:', len(train_mean), '/ test species:', len(test_mean))

## 2. train vs test 平均スペクトル重ね書き

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

for _, row in train_mean.iterrows():
    ax.plot(wavelengths, row[spectral_columns].to_numpy(dtype=float),
            color='tab:orange', alpha=0.55, linewidth=1.2,
            label=f"train: {row['樹種']}")

for _, row in test_mean.iterrows():
    ax.plot(wavelengths, row[spectral_columns].to_numpy(dtype=float),
            color='tab:blue', alpha=0.85, linewidth=1.6, linestyle='--',
            label=f"test: {row['樹種']}")

ax.invert_xaxis()
ax.set_xlabel('Wavenumber (cm$^{-1}$)')
ax.set_ylabel('Absorbance (mean per species)')
ax.set_title('Train (orange, solid) vs Test (blue, dashed) — mean spectra per species')
ax.axvspan(6800, 7200, color='gray', alpha=0.10)
ax.axvspan(5000, 5400, color='gray', alpha=0.10)
ax.text(7000, ax.get_ylim()[1]*0.95, 'OH overtone', ha='center', fontsize=9, color='dimgray')
ax.text(5200, ax.get_ylim()[1]*0.95, 'OH combination', ha='center', fontsize=9, color='dimgray')
ax.legend(loc='upper left', fontsize=7, ncol=2, framealpha=0.85)
fig.tight_layout()
fig.savefig(FIG_DIR / 'mean_spectra_overlay.png', dpi=130)
plt.show()

## 3. PCA で種クラスタを 2D 可視化

train + test の平均スペクトルを縦に結合 → 標準化 → PCA(2) で投影。test 種が train クラスタの内/外を一目で見る。

In [ ]:
train_mean = train_mean.assign(source='train')
test_mean = test_mean.assign(source='test')
all_mean = pd.concat([train_mean, test_mean], axis=0, ignore_index=True)

X = all_mean[spectral_columns].to_numpy(dtype=float)
Xz = StandardScaler().fit_transform(X)
pca = PCA(n_components=2, random_state=0)
Z = pca.fit_transform(Xz)
all_mean['pc1'] = Z[:, 0]
all_mean['pc2'] = Z[:, 1]

fig, ax = plt.subplots(figsize=(9, 7))
for source, marker, color in [('train', 'o', 'tab:orange'), ('test', 'X', 'tab:blue')]:
    sub = all_mean[all_mean['source'] == source]
    ax.scatter(sub['pc1'], sub['pc2'], marker=marker, color=color, s=140,
               edgecolor='black', linewidth=0.5, label=source, alpha=0.85)
    for _, row in sub.iterrows():
        ax.annotate(row['樹種'], (row['pc1'], row['pc2']),
                    xytext=(5, 5), textcoords='offset points', fontsize=9)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title('PCA of per-species mean spectra')
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / 'pca_species_mean.png', dpi=130)
plt.show()

## 4. 各 test 種の最近傍 train 種（cosine similarity）

test 種ごとに、最も似ている train 種を上位 3 つリスト。アンサンブル時に「test 種 X の予測は train 種 Y のモデルを重視」みたいな戦略の根拠になる。

In [ ]:
train_arr = train_mean[spectral_columns].to_numpy(dtype=float)
test_arr = test_mean[spectral_columns].to_numpy(dtype=float)

sim = cosine_similarity(test_arr, train_arr)
sim_df = pd.DataFrame(sim, index=test_mean['樹種'].to_numpy(), columns=train_mean['樹種'].to_numpy())

rows = []
for test_name, row in sim_df.iterrows():
    top = row.sort_values(ascending=False).head(3)
    for rank, (train_name, score) in enumerate(top.items(), start=1):
        rows.append({'test_species': test_name, 'rank': rank, 'nearest_train_species': train_name, 'cosine_sim': float(score)})

nearest_df = pd.DataFrame(rows)
display(nearest_df)
nearest_df.to_csv(FIG_DIR / 'nearest_train_species.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(sim_df, annot=True, fmt='.3f', cmap='viridis', cbar_kws={'label': 'cosine similarity'}, ax=ax)
ax.set_xlabel('train species')
ax.set_ylabel('test species')
ax.set_title('Cosine similarity: test vs train species mean spectra')
fig.tight_layout()
fig.savefig(FIG_DIR / 'similarity_heatmap.png', dpi=130)
plt.show()

## 5. 観察と仮説（実行後に書く）

- [ ] PCA 散布図で test 種が train クラスタの内/外どちらに分布するか
- [ ] cosine similarity が一様に高い (>0.99) なら、種ごとの差は小さい → 全 train データで学習する戦略が正解
- [ ] 特定 test 種だけ類似度が低いなら、その種は外挿になる → 個別対策（強い正則化、SNV/MSC 前処理）の根拠
- [ ] スペクトル重ね書きで、test (青破線) が train (橙) の包絡内にいるか確認